In [1]:
!pip install psycopg2-binary

In [2]:
!pip install kafka-python

In [7]:
import psycopg2
import logging
from kafka import KafkaConsumer
import json

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger(__name__)

# ==============================
# KAFKA CONSUMER — fresh setiap run
# ==============================
consumer = KafkaConsumer(
    "payments",
    bootstrap_servers="kafka:29092",
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    auto_offset_reset="earliest",
    group_id=None,               # ← None agar selalu baca dari awal
    consumer_timeout_ms=10000
)

# ==============================
# KONEKSI POSTGRESQL
# ==============================
conn = psycopg2.connect(
    host="postgres",
    database="olist_ecommerce",
    user="airflow",
    password="airflow",
    port="5432",
    options="-c search_path=raw"
)
cur = conn.cursor()
logger.info("🚀 Consumer jalan...")

# ==============================
# CONSUMER LOOP
# ==============================
total = 0
error_count = 0
BATCH_SIZE = 100
MAX_ERROR = 10

try:
    for message in consumer:
        data = message.value

        try:
            cur.execute("""
                INSERT INTO raw.raw_payments (
                    order_id, payment_sequential, payment_type,
                    payment_installments, payment_value,
                    created_at
                )
                VALUES (%s,%s,%s,%s,%s,%s)
                ON CONFLICT (order_id,payment_sequential) DO NOTHING;
            """, (
                data.get("order_id"),
                data.get("payment_sequential"),
                data.get("payment_type"),
                data.get("payment_installments"),
                data.get("payment_value"),
                data.get("created_at")
            ))
            total += 1

            if total % BATCH_SIZE == 0:
                conn.commit()
                logger.info(f"✅ {total} rows inserted...")

        except Exception as e:
            conn.rollback()
            error_count += 1
            logger.error(f"❌ Error row [{data.get('order_id')}]: {e}")

            if error_count >= MAX_ERROR:
                logger.critical("🚨 Too many errors! Stopping pipeline...")
                raise

except Exception as e:
    logger.critical(f"💀 Pipeline stopped: {e}")
    raise

finally:
    conn.commit()
    cur.close()
    conn.close()
    logger.info(f"🏁 Done. Total {total} rows inserted | {error_count} errors")

2026-03-28 14:47:41,212 - <BrokerConnection client_id=kafka-python-2.3.0, node_id=bootstrap-0 host=kafka:29092 <connecting> [IPv4 ('172.19.0.7', 29092)]>: connecting to kafka:29092 [('172.19.0.7', 29092) IPv4]
2026-03-28 14:47:41,216 - <BrokerConnection client_id=kafka-python-2.3.0, node_id=bootstrap-0 host=kafka:29092 <checking_api_versions_recv> [IPv4 ('172.19.0.7', 29092)]>: Broker version identified as 2.6
2026-03-28 14:47:41,217 - <BrokerConnection client_id=kafka-python-2.3.0, node_id=bootstrap-0 host=kafka:29092 <connected> [IPv4 ('172.19.0.7', 29092)]>: Connection complete.
2026-03-28 14:47:41,217 - group_id is None: disabling auto-commit.
2026-03-28 14:47:41,218 - Updating subscribed topics to: ('payments',)
2026-03-28 14:47:41,223 - 🚀 Consumer jalan...
2026-03-28 14:47:41,225 - Updated partition assignment: [TopicPartition(topic='payments', partition=0)]
2026-03-28 14:47:41,226 - <BrokerConnection client_id=kafka-python-2.3.0, node_id=1 host=kafka:29092 <connecting> [IPv4 ('1